In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style="whitegrid")

df = pd.read_csv('../data/processed/insurance_clean.csv')
print("Shape:", df.shape)
print(df.head())

Shape: (99989, 7)
   age     sex    bmi  children smoker     region   charges
0   55    male  24.70         4     no  northeast  18848.95
1   30    male  20.16         0    yes  southwest  25584.49
2   49  female  29.92         1     no  southeast  18314.86
3   30  female  35.58         0     no  southwest   7256.98
4   20    male  27.15         0    yes  southwest  11595.07


In [2]:
# ── 1. Obesidade ──────────────────────────────────────────
df['is_obese'] = (df['bmi'] > 30).astype(int)

# ── 2. Interação risco máximo: fumante obeso ───────────────
df['smoker_obese'] = ((df['smoker'] == 'yes') & (df['bmi'] > 30)).astype(int)

# ── 3. Faixas etárias usadas em planos de saúde ───────────
bins  = [0, 18, 25, 39, 49, 59, 100]
labels = ['0-18','19-25','26-39','40-49','50-59','60+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

# ── 4. Encoding das categóricas ───────────────────────────
df['smoker_enc']  = (df['smoker'] == 'yes').astype(int)
df['sex_enc']     = (df['sex'] == 'male').astype(int)
df = pd.get_dummies(df, columns=['region'], prefix='region', drop_first=True)
df = pd.get_dummies(df, columns=['age_group'], prefix='age', drop_first=True)

# ── 5. Transformação logarítmica do target ─────────────────
df['log_charges'] = np.log(df['charges'])

# ── 6. Remover colunas originais que já foram encodadas ────
df = df.drop(columns=['sex', 'smoker'])

print("Shape após feature engineering:", df.shape)
print("\nNovas colunas criadas:")
print(df.columns.tolist())

Shape após feature engineering: (99989, 17)

Novas colunas criadas:
['age', 'bmi', 'children', 'charges', 'is_obese', 'smoker_obese', 'smoker_enc', 'sex_enc', 'region_northwest', 'region_southeast', 'region_southwest', 'age_19-25', 'age_26-39', 'age_40-49', 'age_50-59', 'age_60+', 'log_charges']


In [3]:
print("=" * 50)
print("VALIDAÇÃO DAS FEATURES")
print("=" * 50)

print(f"\nis_obese — obesos: {df['is_obese'].sum():,} ({df['is_obese'].mean()*100:.1f}%)")
print(f"smoker_obese — fumantes obesos: {df['smoker_obese'].sum():,} ({df['smoker_obese'].mean()*100:.1f}%)")

print(f"\nCorrelação das novas features com charges:")
novas = ['is_obese','smoker_obese','smoker_enc','sex_enc','log_charges']
for col in novas:
    corr = df[col].corr(df['charges'])
    print(f"  {col:<20} → {corr:.3f}")

print(f"\nCusto médio por grupo de risco:")
grupos = {
    'Não fumante, não obeso' : df[(df['smoker_enc']==0) & (df['is_obese']==0)]['charges'].mean(),
    'Não fumante, obeso'     : df[(df['smoker_enc']==0) & (df['is_obese']==1)]['charges'].mean(),
    'Fumante, não obeso'     : df[(df['smoker_enc']==1) & (df['is_obese']==0)]['charges'].mean(),
    'Fumante, obeso'         : df[(df['smoker_enc']==1) & (df['is_obese']==1)]['charges'].mean(),
}
for grupo, valor in grupos.items():
    print(f"  {grupo:<30} → ${valor:,.0f}")

VALIDAÇÃO DAS FEATURES

is_obese — obesos: 54,376 (54.4%)
smoker_obese — fumantes obesos: 10,849 (10.9%)

Correlação das novas features com charges:
  is_obese             → 0.169
  smoker_obese         → 0.740
  smoker_enc           → 0.816
  sex_enc              → 0.010
  log_charges          → 0.936

Custo médio por grupo de risco:
  Não fumante, não obeso         → $11,322
  Não fumante, obeso             → $13,812
  Fumante, não obeso             → $35,014
  Fumante, obeso                 → $51,462


In [4]:
df.to_csv('../data/processed/insurance_features.csv', index=False)
print("Salvo em data/processed/insurance_features.csv ")
print("Shape final:", df.shape)

Salvo em data/processed/insurance_features.csv 
Shape final: (99989, 17)
